# Hafta 04: Taşıma Katmanı (Transport Layer)
## TCP vs UDP, Güvenilirlik (ARQ), 3 Yönlü El Sıkışma, Checksum ve Akış Kontrolü

> **Ders Kitabı Referansı:** Computer Networking: A Top-Down Approach (J.F. Kurose, K.W. Ross — Bölüm 3)

---

## 1. Temel Kavramlar ve Mimari Çerçeve

> **ÖNEMLİ TERMINOLOJİ NOTU:** Bu bölümde geçen **"Reliable"** kavramı günlük Türkçedeki *şifreli/güvenli (secure)* anlamında değildir; ağ terminolojisinde **"veri kaybına engel olan, paketleri eksiksiz, hatasız ve sırasıyla varış noktasına ulaştıran mekanizmalar bütünü"** anlamındadır.

Ağ (IP) katmanı paketleri **cihazdan cihaza (Host-to-Host)** taşırken; Taşıma Katmanı (TCP/UDP) paketleri cihaz içindeki **uygulamadan uygulamaya (Process-to-Process)** iletir. Bir makinede tek bir IP adresiyle internete çıkan yüzlerce aktif süreci birbirinden ayıran temel mekanizma **Port Numaralarıdır (0 - 65535)**.

---

### 1.1. İki Temel Protokolün Karşılaştırması: TCP vs UDP

| Karşılaştırma Kriteri | UDP (User Datagram Protocol) | TCP (Transmission Control Protocol) |
| :--- | :--- | :--- |
| **Bağlantı Durumu** | **Bağlantısız (Connectionless):** Sunucuya önceden el sıkışma isteği atmaz. | **Bağlantılı (Connection-Oriented):** İletişim öncesi 3-Way Handshake kurar. |
| **Güvenilirlik (Reliability)** | **Unreliable:** Paket kaybını takip etmez, kaybolan paketi yeniden istemez. | **Reliable:** Sıra no (Seq), Onay (ACK) ve Zamanlayıcı (Timeout) ile kayıpsız iletim garantisi verir. |
| **Akış ve Tıkanıklık Kontrolü** | Yoktur; uygulama ne hızda üretirse hatta basar. | Vardır; alıcı tamponunu taşırmaz (*Flow Control*), ağdaki router'ları tıkamaz (*Congestion Control*). |
| **Protokol Ek Yükü (Header)** | Sabit **8 Bayt** (Hafif ve minimal). | Standart **20 Bayt** (Seçeneklerle 60 bayta kadar çıkabilir). |
| **İletim Biçimi** | Teke-tek (Unicast) ve Herkese Yayma (**Broadcast / Multicast**). | Yalnızca noktadan noktaya (Point-to-Point, tekil 1'e 1 akış). |
| **Kullanım Alanları** | DNS, DHCP, RIP, SNMP, canlı ses/video (VoIP), online oyunlar. | HTTP/HTTPS (Web), SMTP (E-posta), FTP (Dosya transferi), SSH. |

---

### 1.2. Neden TCP Yerine UDP Tercih Edilir?

TCP'nin tüm güvenilirlik garantilerine rağmen geliştiricilerin özellikle UDP seçmesinin 4 teknik nedeni vardır:
1. **Hassas Zamanlama Kontrolü:** TCP tıkanıklık anında pencereyi daraltır ve veri iletimini yavaşlatır; üstelik kaybolan paket için ACK gelene dek sonraki verileri bekletir (*Head-of-Line Blocking*). Canlı yayında veya oyunda 2 saniye önceki karenin tekrar gelmesinin anlamı yoktur; veri kaybı tolere edilir ancak gecikme (lag) tolere edilemez.
2. **Bağlantı Kurma Gecikmesinin (RTT) Olmaması:** TCP'nin 3 adımlı el sıkışması ek bir Round-Trip Time (RTT) gecikmesi yaratır. DNS sorgusu için önce TCP el sıkışması beklemek web sayfalarını aşırı yavaşlatacağından DNS doğrudan UDP ile çalışır.
3. **Bağlantı Durumu Bellek Yükünün Olmaması:** TCP sunucusu bağlı her istemci için bellek tamponları (*Send/Receive Buffers*), Seq/ACK sayaçları ve zamanlayıcılar saklar. UDP durum tutmadığı için tek bir sunucu aynı anda yüz binlerce istemciye çok daha az RAM ile hizmet verir.
4. **Başlık Ek Yükünün Azlığı (8 Bayt vs 20 Bayt):** Her pakette 12 baytlık başlık tasarrufu, IoT cihazlarında ve yüksek frekanslı mesajlaşmada devasa bant genişliği ve pil kazancı sağlar.

> **SORU: Aynı makinede hem TCP hem UDP aynı port numarasını (Örn: Port 53) kullanabilir mi?**  
> **CEVAP:** Evet. İşletim sistemi çekirdeği portları protokol bazında ayrı tablolarda tutar. Bir soketin tekilliği 4 parametre ile belirlenir: `(Kaynak IP, Kaynak Port, Hedef IP, Hedef Port)` ve Taşıma Protokolü (`TCP` veya `UDP`). Nitekim DNS servisi hem UDP 53'ü (hızlı sorgular) hem TCP 53'ü (büyük bölge transferleri - Zone Transfer) aynı anda dinler.

---

### 1.3. Güvenilir Veri İletiminin (ARQ) Evrimi ve Performans Darboğazı

Kayıplı ve bozuk paket barındıran gerçek dünya ağlarında güvenilirlik sağlamak için **Otomatik Tekrar İsteme (ARQ - Automatic Repeat reQuest)** algoritmaları geliştirilmiştir:

1. **Stop-and-Wait ARQ (Dur ve Bekle):** Gönderici 1 paket atar, kronometreyi başlatır ve bekler (*Stop & Wait*). Karşıdan `ACK` gelirse bir sonraki paketi atar. Eğer zaman aşımı (**Timeout - kabaca RTT süresi**) dolarsa paketi tekrar yollar.
2. **Stop-and-Wait Performans Faciası (Kullanım Oranı - Utilization Analizi):**
   * Hat Kapasitesi ($R$): 1 Gbps ($10^9$ bps)
   * Tek Yönlü Yayılım Gecikmesi: 15 ms $\rightarrow$ Gidiş-Dönüş Süresi ($RTT$): 30 ms ($0.030$ s)
   * Paket Boyutu ($L$): 1000 Bayt = 8000 bit
   * İletim Süresi: $d_{\text{trans}} = \frac{L}{R} = \frac{8000}{10^9} = 8\ \mu\text{s} = 0.000008\ \text{saniye}$
   
   $$\text{Kullanım Oranı } (U) = \frac{d_{\text{trans}}}{RTT + d_{\text{trans}}} = \frac{0.000008}{0.030008} \approx 0.000267\ (\%0.027)$$
   
   1 Gbps'lik bir omurga hattında Stop-and-Wait kullanıldığında efektif aktarım hızı yalnızca **267 kbps** seviyesine düşer; devasa hattın %99.97'si boşta ACK bekleyerek heba edilir!
3. **Modern Çözüm: Boru Hattı (Pipelining - GBN ve Selective Repeat):** Tek tek beklemek yerine, ACK beklemeden pencere boyutu ($N$) kadar paketi peş peşe hatta basmaktır. TCP bu mantıkla çalışır.

---
## 2. Mimari Görselleştirmeler (Vektörel SVG)

Aşağıdaki kod hücreleri; UDP vs TCP Başlık Formatlarını, Stop-and-Wait ARQ vs Pipelining Mantığını ve TCP 3 Yönlü El Sıkışma (3-Way Handshake) sürecini saf vektörel SVG olarak ekrana çizer.

In [ ]:
from IPython.display import SVG, display

# Şekil 1: UDP (8B) vs TCP (20B) Segment Başlık Yapısı Karşılaştırması
svg_headers = """
<svg width="820" height="360" viewBox="0 0 820 360" xmlns="http://www.w3.org/2000/svg" style="background:#ffffff; font-family:sans-serif;">
  <text x="410" y="24" text-anchor="middle" font-size="14" font-weight="bold" fill="#111827">
    Kurose &amp; Ross Taşıma Katmanı Başlık Mimarisi: UDP (8 Bayt) vs TCP (20 Bayt)
  </text>

  <!-- SOL: UDP BAŞLIĞI (8 Bayt) -->
  <g transform="translate(40, 50)">
    <rect width="340" height="280" rx="8" fill="#f8f9fa" stroke="#cbd5e1" stroke-width="1.5"/>
    <text x="170" y="26" text-anchor="middle" font-size="12" font-weight="bold" fill="#0369a1">UDP Segment Başlığı (Sabit 8 Bayt)</text>
    
    <!-- 1. Satır -->
    <rect x="20" y="45" width="145" height="50" fill="#e0f2fe" stroke="#0284c7" stroke-width="1.5"/>
    <text x="92" y="68" text-anchor="middle" font-size="10" font-weight="bold" fill="#0369a1">Kaynak Port (16 bit)</text>
    <text x="92" y="84" text-anchor="middle" font-size="9" fill="#075985">Source Port (2B)</text>

    <rect x="175" y="45" width="145" height="50" fill="#e0f2fe" stroke="#0284c7" stroke-width="1.5"/>
    <text x="247" y="68" text-anchor="middle" font-size="10" font-weight="bold" fill="#0369a1">Hedef Port (16 bit)</text>
    <text x="247" y="84" text-anchor="middle" font-size="9" fill="#075985">Dest Port (2B)</text>

    <!-- 2. Satır -->
    <rect x="20" y="105" width="145" height="50" fill="#e0f2fe" stroke="#0284c7" stroke-width="1.5"/>
    <text x="92" y="128" text-anchor="middle" font-size="10" font-weight="bold" fill="#0369a1">Segment Boyutu (16 bit)</text>
    <text x="92" y="144" text-anchor="middle" font-size="9" fill="#075985">Length: Header+Data (2B)</text>

    <rect x="175" y="105" width="145" height="50" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/>
    <text x="247" y="128" text-anchor="middle" font-size="10" font-weight="bold" fill="#92400e">Checksum (16 bit)</text>
    <text x="247" y="144" text-anchor="middle" font-size="9" fill="#b45309">Hata Kontrolü (2B)</text>

    <!-- Payload -->
    <rect x="20" y="165" width="300" height="85" rx="4" fill="#f1f5f9" stroke="#94a3b8" stroke-width="1.5" stroke-dasharray="3,3"/>
    <text x="170" y="202" text-anchor="middle" font-size="11" font-weight="bold" fill="#475569">Uygulama Verisi (Payload)</text>
    <text x="170" y="220" text-anchor="middle" font-size="9" fill="#64748b">DNS Sorgusu, Ses Verisi, Oyun Pozisyonu vb.</text>
  </g>

  <!-- SAĞ: TCP BAŞLIĞI (20 Bayt) -->
  <g transform="translate(420, 50)">
    <rect width="360" height="280" rx="8" fill="#f8f9fa" stroke="#cbd5e1" stroke-width="1.5"/>
    <text x="180" y="26" text-anchor="middle" font-size="12" font-weight="bold" fill="#15803d">TCP Segment Başlığı (Temel 20 Bayt)</text>
    
    <!-- Portlar -->
    <rect x="15" y="42" width="160" height="36" fill="#dcfce7" stroke="#16a34a" stroke-width="1"/>
    <text x="95" y="64" text-anchor="middle" font-size="9" font-weight="bold" fill="#14532d">Kaynak Port (16b)</text>
    <rect x="185" y="42" width="160" height="36" fill="#dcfce7" stroke="#16a34a" stroke-width="1"/>
    <text x="265" y="64" text-anchor="middle" font-size="9" font-weight="bold" fill="#14532d">Hedef Port (16b)</text>

    <!-- Sequence Number -->
    <rect x="15" y="82" width="330" height="34" fill="#dbeafe" stroke="#2563eb" stroke-width="1"/>
    <text x="180" y="103" text-anchor="middle" font-size="10" font-weight="bold" fill="#1e40af">Sıra Numarası: Sequence Number (32 bit / 4 Bayt)</text>

    <!-- ACK Number -->
    <rect x="15" y="120" width="330" height="34" fill="#dbeafe" stroke="#2563eb" stroke-width="1"/>
    <text x="180" y="141" text-anchor="middle" font-size="10" font-weight="bold" fill="#1e40af">Onay Numarası: Acknowledgment Number (32 bit / 4 Bayt)</text>

    <!-- Bayraklar ve Pencere -->
    <rect x="15" y="158" width="65" height="36" fill="#fef3c7" stroke="#d97706" stroke-width="1"/>
    <text x="47" y="180" text-anchor="middle" font-size="8" font-weight="bold" fill="#92400e">HLEN (4b)</text>

    <rect x="85" y="158" width="135" height="36" fill="#fee2e2" stroke="#dc2626" stroke-width="1"/>
    <text x="152" y="175" text-anchor="middle" font-size="8" font-weight="bold" fill="#991b1b">Bayraklar (Flags: 6b)</text>
    <text x="152" y="188" text-anchor="middle" font-size="7" fill="#b91c1c">[URG, ACK, PSH, RST, SYN, FIN]</text>

    <rect x="225" y="158" width="120" height="36" fill="#ede9fe" stroke="#7c3aed" stroke-width="1"/>
    <text x="285" y="180" text-anchor="middle" font-size="8" font-weight="bold" fill="#5b21b6">Receive Window (16b)</text>

    <!-- Checksum ve Acil -->
    <rect x="15" y="198" width="160" height="34" fill="#fef3c7" stroke="#d97706" stroke-width="1"/>
    <text x="95" y="219" text-anchor="middle" font-size="9" font-weight="bold" fill="#92400e">Checksum (16 bit)</text>
    <rect x="185" y="198" width="160" height="34" fill="#f1f5f9" stroke="#94a3b8" stroke-width="1"/>
    <text x="265" y="219" text-anchor="middle" font-size="9" fill="#64748b">Urgent Pointer (16 bit)</text>

    <!-- Payload -->
    <rect x="15" y="236" width="330" height="32" fill="#f1f5f9" stroke="#94a3b8" stroke-width="1" stroke-dasharray="3,3"/>
    <text x="180" y="256" text-anchor="middle" font-size="9" font-weight="bold" fill="#475569">Uygulama Yükü (HTTP, TLS, Dosya Baytları)</text>
  </g>
</svg>
"""
display(SVG(svg_headers))

In [ ]:
# Şekil 2: Stop-and-Wait ARQ (%0.027 Verim) vs Boru Hattı (Pipelining / GBN / Selective Repeat)
svg_arq = """
<svg width="820" height="300" viewBox="0 0 820 300" xmlns="http://www.w3.org/2000/svg" style="background:#ffffff; font-family:sans-serif;">
  <defs>
    <marker id="arr-b" markerWidth="6" markerHeight="6" refX="5" refY="3" orient="auto">
      <polygon points="0 0, 6 3, 0 6" fill="#2563eb"/>
    </marker>
    <marker id="arr-g" markerWidth="6" markerHeight="6" refX="5" refY="3" orient="auto">
      <polygon points="0 0, 6 3, 0 6" fill="#16a34a"/>
    </marker>
  </defs>
  <text x="410" y="24" text-anchor="middle" font-size="14" font-weight="bold" fill="#111827">
    Güvenilirlik Modelleri: Stop-and-Wait ARQ (At ve Bekle) vs Boru Hattı (Pipelining / Pencere)
  </text>

  <!-- SOL: STOP-AND-WAIT -->
  <g transform="translate(30, 45)">
    <rect width="360" height="240" rx="8" fill="#fff1f2" stroke="#f43f5e" stroke-width="1.5"/>
    <text x="180" y="24" text-anchor="middle" font-size="12" font-weight="bold" fill="#9f1239">Stop-and-Wait ARQ (Verim: %0.027)</text>
    <line x1="70" y1="50" x2="70" y2="220" stroke="#64748b" stroke-width="1.5" stroke-dasharray="3,3"/>
    <line x1="290" y1="50" x2="290" y2="220" stroke="#64748b" stroke-width="1.5" stroke-dasharray="3,3"/>
    <text x="70" y="44" text-anchor="middle" font-size="10" font-weight="bold" fill="#334155">Gönderici</text>
    <text x="290" y="44" text-anchor="middle" font-size="10" font-weight="bold" fill="#334155">Alıcı</text>

    <!-- 1. Paket ve ACK -->
    <line x1="70" y1="70" x2="285" y2="115" stroke="#2563eb" stroke-width="2" marker-end="url(#arr-b)"/>
    <text x="170" y="86" text-anchor="middle" font-size="9" fill="#1e40af">Paket #0 (L/R = 8 µs)</text>
    <line x1="290" y1="120" x2="75" y2="165" stroke="#16a34a" stroke-width="2" marker-end="url(#arr-g)"/>
    <text x="185" y="140" text-anchor="middle" font-size="9" fill="#15803d">ACK #0</text>

    <!-- Boş Bekleme Zamanı -->
    <text x="180" y="190" text-anchor="middle" font-size="10" font-weight="bold" fill="#be123c">
      30 ms RTT Boyunca Hat Boş Bekliyor!
    </text>
    <line x1="70" y1="195" x2="285" y2="225" stroke="#2563eb" stroke-width="1.5" marker-end="url(#arr-b)"/>
    <text x="170" y="206" text-anchor="middle" font-size="8" fill="#1e40af">Paket #1</text>
  </g>

  <!-- SAĞ: PIPELINING -->
  <g transform="translate(430, 45)">
    <rect width="360" height="240" rx="8" fill="#f0fdf4" stroke="#22c55e" stroke-width="1.5"/>
    <text x="180" y="24" text-anchor="middle" font-size="12" font-weight="bold" fill="#14532d">Boru Hattı (Pipelining - N=3 Pencere)</text>
    <line x1="70" y1="50" x2="70" y2="220" stroke="#64748b" stroke-width="1.5" stroke-dasharray="3,3"/>
    <line x1="290" y1="50" x2="290" y2="220" stroke="#64748b" stroke-width="1.5" stroke-dasharray="3,3"/>
    <text x="70" y="44" text-anchor="middle" font-size="10" font-weight="bold" fill="#334155">Gönderici</text>
    <text x="290" y="44" text-anchor="middle" font-size="10" font-weight="bold" fill="#334155">Alıcı</text>

    <!-- 3 Paket Peş Peşe -->
    <line x1="70" y1="65" x2="285" y2="100" stroke="#2563eb" stroke-width="1.5" marker-end="url(#arr-b)"/>
    <line x1="70" y1="80" x2="285" y2="115" stroke="#2563eb" stroke-width="1.5" marker-end="url(#arr-b)"/>
    <line x1="70" y1="95" x2="285" y2="130" stroke="#2563eb" stroke-width="1.5" marker-end="url(#arr-b)"/>
    <text x="165" y="78" text-anchor="middle" font-size="8" fill="#1e40af">Paket 0, 1, 2 Peş Peşe Basılır</text>

    <!-- Dönüş ACK'leri -->
    <line x1="290" y1="105" x2="75" y2="140" stroke="#16a34a" stroke-width="1.5" marker-end="url(#arr-g)"/>
    <line x1="290" y1="120" x2="75" y2="155" stroke="#16a34a" stroke-width="1.5" marker-end="url(#arr-g)"/>
    <line x1="290" y1="135" x2="75" y2="170" stroke="#16a34a" stroke-width="1.5" marker-end="url(#arr-g)"/>
    <text x="180" y="162" text-anchor="middle" font-size="8" fill="#15803d">ACK 0, 1, 2 Sırayla Döner</text>

    <text x="180" y="205" text-anchor="middle" font-size="10" font-weight="bold" fill="#15803d">
      Hat Kullanımı N Katına Çıkar (Verim %300 Artar!)
    </text>
  </g>
</svg>
"""
display(SVG(svg_arq))

In [ ]:
# Şekil 3: TCP 3 Yönlü El Sıkışma (3-Way Handshake) ve MSS/MTU Ayrımı
svg_handshake = """
<svg width="820" height="310" viewBox="0 0 820 310" xmlns="http://www.w3.org/2000/svg" style="background:#ffffff; font-family:sans-serif;">
  <defs>
    <marker id="arr-syn" markerWidth="7" markerHeight="7" refX="5" refY="3.5" orient="auto">
      <polygon points="0 0, 7 3.5, 0 7" fill="#dc2626"/>
    </marker>
    <marker id="arr-sa" markerWidth="7" markerHeight="7" refX="5" refY="3.5" orient="auto">
      <polygon points="0 0, 7 3.5, 0 7" fill="#d97706"/>
    </marker>
    <marker id="arr-ack" markerWidth="7" markerHeight="7" refX="5" refY="3.5" orient="auto">
      <polygon points="0 0, 7 3.5, 0 7" fill="#16a34a"/>
    </marker>
  </defs>
  <text x="410" y="24" text-anchor="middle" font-size="14" font-weight="bold" fill="#111827">
    TCP 3 Yönlü El Sıkışma (Three-Way Handshake) Protokol Diyaloğu
  </text>

  <rect x="90" y="45" width="130" height="40" rx="6" fill="#dbeafe" stroke="#1d4ed8" stroke-width="1.5"/>
  <text x="155" y="69" text-anchor="middle" font-size="11" font-weight="bold" fill="#1e40af">İSTEMCİ (Client)</text>
  <line x1="155" y1="85" x2="155" y2="290" stroke="#94a3b8" stroke-width="1.5" stroke-dasharray="4,4"/>

  <rect x="590" y="45" width="140" height="40" rx="6" fill="#fee2e2" stroke="#dc2626" stroke-width="1.5"/>
  <text x="660" y="69" text-anchor="middle" font-size="11" font-weight="bold" fill="#991b1b">SUNUCU (Listen: 80)</text>
  <line x1="660" y1="85" x2="660" y2="290" stroke="#94a3b8" stroke-width="1.5" stroke-dasharray="4,4"/>

  <!-- 1. Adım: SYN -->
  <line x1="155" y1="115" x2="650" y2="150" stroke="#dc2626" stroke-width="2" marker-end="url(#arr-syn)"/>
  <text x="405" y="125" text-anchor="middle" font-size="10" font-weight="bold" fill="#b91c1c">
    1. ADIM: SYN=1, Seq=x (Payload = 0B)
  </text>
  <text x="405" y="140" text-anchor="middle" font-size="9" fill="#7f1d1d">
    "Bağlanmak istiyorum, benim başlangıç sıra numaram x"
  </text>

  <!-- 2. Adım: SYN-ACK -->
  <line x1="660" y1="170" x2="165" y2="205" stroke="#d97706" stroke-width="2" marker-end="url(#arr-sa)"/>
  <text x="405" y="180" text-anchor="middle" font-size="10" font-weight="bold" fill="#b45309">
    2. ADIM: SYN=1, ACK=1, Seq=y, Ack=x+1 (Payload = 0B)
  </text>
  <text x="405" y="195" text-anchor="middle" font-size="9" fill="#78350f">
    "İsteğini kabul ettim (x+1), benim sıra numaram y"
  </text>

  <!-- 3. Adım: ACK (+ Opsiyonel Payload) -->
  <line x1="155" y1="225" x2="650" y2="260" stroke="#16a34a" stroke-width="2" marker-end="url(#arr-ack)"/>
  <text x="405" y="235" text-anchor="middle" font-size="10" font-weight="bold" fill="#15803d">
    3. ADIM: ACK=1, Seq=x+1, Ack=y+1 (+ HTTP GET Verisi Taşınabilir)
  </text>
  <text x="405" y="250" text-anchor="middle" font-size="9" fill="#14532d">
    "Onayını aldım (y+1), oturum açıldı! Veri transferi başlıyor."
  </text>

  <!-- MSS Notu -->
  <text x="410" y="285" text-anchor="middle" font-size="10" fill="#475569" font-style="italic">
    Ethernet MTU = 1500B  ==&gt;  MSS = 1500 - TCP_Header(20B) - IP_Header(20B) = 1460 Bayt azami veri.
  </text>
</svg>
"""
display(SVG(svg_handshake))

---
## 3. Deney 1: UDP Checksum Hesaplama ve 1-Bit Bozulma (Bit Error) Tespiti

### 3.1. Deneyin Amacı
Ders notlarında belirtilen; UDP protokolünün sunduğu yegâne hata denetim servisi olan **16-bit İnternet Checksum (1'lik Tümleyen Toplamı)** algoritmasını sıfırdan saf Python ile kodlamak; kabloda meydana gelen 1-bitlik bir elektriksel bozulmanın alıcı tarafından nasıl anında yakalandığını somut olarak gözlemlemektir.

In [ ]:
# ==============================================================================
# DENEY 1: Standart 16-Bit İnternet Checksum Algoritması ve Hata Tespiti
# ==============================================================================

def calculate_checksum(data: bytes) -> int:
    """
    RFC 1071 standardına göre 16-bit 1'lik tümleyen sağlama toplamı hesaplar.
    """
    if len(data) % 2 == 1:
        data += b'\x00'  # Tek bayt ise sonuna sıfır doldur (Padding)

    toplam = 0
    for i in range(0, len(data), 2):
        word16 = (data[i] << 8) + data[i + 1]
        toplam += word16
        # 16-bit taşması (Overflow) olursa eldeyi en alt bite ekle (End-around carry)
        while (toplam >> 16) > 0:
            toplam = (toplam & 0xFFFF) + (toplam >> 16)

    # Sonucun 1'lik tümleyenini (Invert) al
    checksum = ~toplam & 0xFFFF
    return checksum

def verify_checksum(received_data: bytes, received_checksum: int) -> bool:
    """
    Alıcı taraf: Veri ile gelen checksum toplandığında sonuç 0xFFFF (tüm bitler 1) çıkmalıdır.
    """
    if len(received_data) % 2 == 1:
        received_data += b'\x00'

    toplam = 0
    for i in range(0, len(received_data), 2):
        word16 = (received_data[i] << 8) + received_data[i + 1]
        toplam += word16
        while (toplam >> 16) > 0:
            toplam = (toplam & 0xFFFF) + (toplam >> 16)

    toplam += received_checksum
    while (toplam >> 16) > 0:
        toplam = (toplam & 0xFFFF) + (toplam >> 16)

    return (toplam == 0xFFFF)

# --- TEST VE SİMÜLASYON ---
orijinal_mesaj = b"MCBU_NETLAB_ODEV_1"
gonderilen_cs = calculate_checksum(orijinal_mesaj)

print("=" * 65)
print("UDP CHECKSUM SAĞLAMA VE HATA TESPİT SİMÜLASYONU")
print("=" * 65)
print(f"Orijinal Yük        : {orijinal_mesaj}")
print(f"Hesaplanan Checksum : 0x{gonderilen_cs:04X} ({bin(gonderilen_cs)})")

# Senaryo A: Paket Kablodan Hatasız Ulaştı
durum_hatasiz = verify_checksum(orijinal_mesaj, gonderilen_cs)
print(f"\n[TEST 1] Hatasız İletim Doğrulaması : {'GEÇTİ (Sağlam)' if durum_hatasiz else 'BAŞARISIZ'}")

# Senaryo B: Hatta Elektriksel Gürültü Oldu (Tek bir karakter bozuldu: '1' -> '2')
bozuk_mesaj = b"MCBU_NETLAB_ODEV_2"
durum_bozuk = verify_checksum(bozuk_mesaj, gonderilen_cs)
print(f"[TEST 2] 1-Bit Bozulma Doğrulaması  : {'GEÇTİ (Sağlam)' if durum_bozuk else 'YAKALANDI (Bozuk Paket Düşürüldü!)'}")
print("=" * 65)

---
### 3.2. Deney 1'in Mühendislik Yorumu
1. **UDP Hata Düzeltmez, Yalnızca Tespit Eder:** UDP alıcısı checksum uyuşmazlığı yakaladığında paketi anında çöpe atar (*DROP*). Göndericiye haber vermez, paketi tekrar istemez. Hata düzeltme gerekiyorsa uygulama katmanı kendi mekanizmasını yazmalı veya doğrudan TCP seçilmelidir.
2. **End-Around Carry (Elde Ekleme):** 16-bit toplama sırasında en soldan taşan bitler atılmaz; en sağdaki en değersiz bite (LSB) eklenir. Bu kural, mimariden bağımsız olarak (Big-Endian vs Little-Endian) checksum tutarlılığını garanti eder.

---
## 4. Deney 2: Stop-and-Wait ARQ Simülatörü (Timeout ve Kayıp Paket Analizi)

### 4.1. Deneyin Amacı
Ders notlarında açıklanan; Stop-and-Wait ARQ algoritmasının hat üzerinde kaybolan bir paketi zamanlayıcı (**Timeout**) geri sayımıyla nasıl yakaladığını, sıra numaraları (`Seq 0` ve `Seq 1`) ile kopyalanmış mükerrer paketleri nasıl ayırt ettiğini simüle etmektir.

In [ ]:
# ==============================================================================
# DENEY 2: Stop-and-Wait ARQ (Zaman Aşımı ve Sıra Numarası Kontrolü)
# ==============================================================================
import random
import time

class StopAndWaitARQSimulator:
    def __init__(self, paket_kayip_olasiligi=0.3):
        self.loss_rate = paket_kayip_olasiligi
        self.expected_seq_at_receiver = 0

    def transmit_channel(self, item_name: str):
        """Kayıplı ağ ortamını simüle eder"""
        if random.random() < self.loss_rate:
            return False  # Paket yolda kayboldu
        return True       # Paket ulaştı

    def send_packets(self, packet_list: list):
        print("=" * 65)
        print(f"STOP-AND-WAIT ARQ SİMÜLASYONU (Kayıp İhtimali: %{int(self.loss_rate*100)})")
        print("=" * 65)
        
        current_seq = 0
        for pkt_data in packet_list:
            ack_received = False
            deneme_sayisi = 1
            
            while not ack_received:
                print(f"\n[GÖNDERİCİ] -> Paket (Seq={current_seq}, Data='{pkt_data}') Hatta Verildi (Deneme #{deneme_sayisi})")
                
                # 1. Hatta Paketin İletimi
                if not self.transmit_channel("DATA"):
                    print(f"  [!] AĞ DARBOĞAZI: Paket (Seq={current_seq}) Router kuyruğunda DÜŞTÜ!")
                    print(f"  [!] ZAMANLAYICI DOLDU: Timeout (RTT süresi bitti)! Paket yeniden iletilecek...")
                    deneme_sayisi += 1
                    continue
                
                # 2. Alıcı Tarafın İşlemesi
                print(f"  [ALICI] <- Paket (Seq={current_seq}) Hatasız Ulaştı.")
                
                # 3. ACK Mesajının Dönüşü
                if not self.transmit_channel("ACK"):
                    print(f"  [!] DÖNÜŞ HATTI: ACK #{current_seq} yolda KAYBOLDU!")
                    print(f"  [!] GÖNDERİCİ TIMEOUT: ACK gelmediği için aynı paketi tekrar basacak!")
                    deneme_sayisi += 1
                    continue
                
                # ACK Başarıyla Ulaştı
                print(f"[GÖNDERİCİ] <- ACK #{current_seq} Başarıyla Alındı. Sıradaki pakete geçilebilir.")
                ack_received = True
                current_seq = 1 - current_seq  # Sıra numarasını değiştir (0 -> 1 -> 0)

# Simülasyonu Çalıştır
random.seed(42)  # Tekrarlanabilir çıktı için
sim = StopAndWaitARQSimulator(paket_kayip_olasiligi=0.35)
sim.send_packets(["VERI_A", "VERI_B", "VERI_C"])

---
### 4.2. Deney 2'nin Mühendislik Yorumu
1. **ACK Kaybolduğunda Mükerrer (Duplicate) Paket Sorunu:** Eğer giden veri değil de dönen ACK kaybolursa, gönderici zaman aşımına uğrayıp aynı paketi bir daha atar. Alıcı taraf gelen paketin üzerindeki **Sıra Numarasına (Seq 0/1)** bakarak onun kopya olduğunu anlar; veriyi uygulamaya tekrar kopyalamaz, sadece kaybolan ACK'yi yeniden gönderir.
2. **Timeout Süresinin Kritik Önemi:** Zaman aşımı süresi RTT'den çok kısa seçilirse henüz yolda olan paketler için gereksiz yere mükerrer paket basılır (ağ felç olur). Çok uzun seçilirse kaybolan paketin fark edilmesi saniyeler sürer ve iletim duraklar. TCP bu süreyi ağdaki anlık dalgalanmalara göre dinamik (*Adaptive Retransmission Timeout*) hesaplar.

---
## 5. Deney 3: Scapy ile TCP 3 Yönlü El Sıkışma (SYN -> SYN-ACK -> ACK) Analizi

### 5.1. Deneyin Amacı
TCP'nin güvenilirlik temeli olan **3-Way Handshake** sürecindeki bayrak bitlerini (`SYN`, `ACK`), 32-bitlik rastgele başlangıç Sequence numaralarını ve bunların onaylanma matematiğini Scapy ile paket bazında inşa ederek doğrulamaktır.

In [ ]:
# ==============================================================================
# DENEY 3: Scapy ile TCP 3-Way Handshake Paket Katarı İnşası ve Doğrulaması
# ==============================================================================
import logging
logging.getLogger("scapy.runtime").setLevel(logging.ERROR)

from scapy.all import Ether, IP, TCP, Raw, wrpcap, rdpcap

# Başlangıç Parametreleri
CLIENT_IP = "192.168.1.50"
SERVER_IP = "93.184.216.34"  # example.com
CLIENT_PORT = 54321
SERVER_PORT = 80

client_isn = 100000  # İstemcinin İlk Sıra Numarası (Initial Sequence Number - ISN)
server_isn = 500000  # Sunucunun İlk Sıra Numarası

# 1. ADIM: İSTEMCİ -> SUNUCU (SYN)
syn_pkt = (
    Ether(src="AA:AA:AA:AA:AA:AA", dst="BB:BB:BB:BB:BB:BB") /
    IP(src=CLIENT_IP, dst=SERVER_IP) /
    TCP(sport=CLIENT_PORT, dport=SERVER_PORT, flags="S", seq=client_isn, ack=0)
)

# 2. ADIM: SUNUCU -> İSTEMCİ (SYN-ACK)
# Sunucu istemcinin ISN değerini 1 artırarak onaylar (ack = client_isn + 1)
syn_ack_pkt = (
    Ether(src="BB:BB:BB:BB:BB:BB", dst="AA:AA:AA:AA:AA:AA") /
    IP(src=SERVER_IP, dst=CLIENT_IP) /
    TCP(sport=SERVER_PORT, dport=CLIENT_PORT, flags="SA", seq=server_isn, ack=client_isn + 1)
)

# 3. ADIM: İSTEMCİ -> SUNUCU (ACK)
# İstemci sunucunun ISN değerini 1 artırarak el sıkışmayı tamamlar (ack = server_isn + 1)
ack_pkt = (
    Ether(src="AA:AA:AA:AA:AA:AA", dst="BB:BB:BB:BB:BB:BB") /
    IP(src=CLIENT_IP, dst=SERVER_IP) /
    TCP(sport=CLIENT_PORT, dport=SERVER_PORT, flags="A", seq=client_isn + 1, ack=server_isn + 1)
)

# Paket katarını diske yazalım
pcap_dosyasi = "tcp_3way_handshake.pcap"
wrpcap(pcap_dosyasi, [syn_pkt, syn_ack_pkt, ack_pkt])
print(f"[+] TCP 3-Way Handshake oturumu '{pcap_dosyasi}' dosyasına kaydedildi.\n")

# Dosyadan okuyup bayrak ve sayaç matematiğini denetleyelim
oturum = rdpcap(pcap_dosyasi)

print("=" * 70)
print("TCP 3 YÖNLÜ EL SIKIŞMA BAŞLIK VE BAYRAK ANALİZİ")
print("=" * 70)
for i, p in enumerate(oturum, start=1):
    flags = p[TCP].flags
    flags_str = "SYN" if flags == "S" else ("SYN-ACK" if flags == "SA" else "ACK")
    print(f"ADIM #{i} [{flags_str:<7}] -> Seq: {p[TCP].seq:<7} | Ack: {p[TCP].ack:<7} | Flags: {p[TCP].flags}")
print("=" * 70)

---
### 5.2. Deney 3'ün Mühendislik Yorumu
1. **Hayali Bayt Tüketimi (Phantom Byte):** İlk iki adımda (`SYN` ve `SYN-ACK`) hiçbir uygulama verisi (Payload = 0 bayt) taşınmaz. Buna rağmen karşı taraf onay numarasını (`Ack`) 1 artırır (`100000 -> 100001`). TCP standardında bağlantı kurma bayrakları (`SYN`, `FIN`) mantıksal olarak 1 bayt veri tüketmiş gibi işlem görür.
2. **Neden 2 Adım Yetmez de 3 Adım Gerekir?** İstemci bağlanmak istediğinde sunucu onay verirse sunucu hazır olur; fakat sunucunun kendi Seq numarasını istemcinin alıp almadığını sunucunun bilmesi gerekir. 3. adım (`ACK`) atılmazsa sunucu istemcinin hatta olup olmadığını bilemez ve askıda kalan bağlantılarla bellek tükenir (*SYN Flood zafiyeti*).

---
## 6. Laboratuvar Uygulaması: TCP Akış Kontrolü (Flow Control) ve Alıcı Tamponu (Receive Window) Simülasyonu

### 6.1. Uygulamanın Amacı
TCP'nin alıcı makineyi korumak için kullandığı **Akış Kontrolü (Flow Control)** mekanizmasını modellemek; alıcının tüketim hızına göre TCP başlığındaki **Receive Window (rwnd)** alanını nasıl dinamik güncellediğini ve tampon dolduğunda (`rwnd = 0`) göndericiyi nasıl durdurduğunu bizzat kodlayarak doğrulamaktır.

### 6.2. Yönerge
1. `TCPReceiverBuffer` sınıfı oluşturun (Maksimum tampon kapasitesi: **4096 Bayt**).
2. Gönderici hızlı şekilde **1024 baytlık** segmentler fırlatsın.
3. Alıcı uygulama veriyi yavaş tüketsin (Örneğin her adımda sadece 256 bayt okunsun).
4. Tamponda kalan boş alanı her adımda hesaplayıp TCP başlığındaki `rwnd` değeri olarak göndericiye dönün.
5. `rwnd == 0` olduğu anda göndericinin iletimi durdurduğunu konsola formatlı dökün.

In [ ]:
# ==============================================================================
# UYGULAMA: TCP Akış Kontrolü ve Dinamik Receive Window (rwnd) Simülasyonu
# ==============================================================================

class TCPReceiverBuffer:
    def __init__(self, max_buffer_size=4096):
        self.max_buffer = max_buffer_size
        self.current_data_size = 0

    def receive_segment(self, segment_payload_size: int) -> int:
        """Gelen segmenti tampona yazar ve güncel rwnd değerini döner"""
        self.current_data_size += segment_payload_size
        rwnd = max(0, self.max_buffer - self.current_data_size)
        return rwnd

    def application_read(self, read_size: int):
        """Alıcı işletim sistemindeki yazılım sürecinin tampondan veri okuması"""
        self.current_data_size = max(0, self.current_data_size - read_size)
        rwnd = max(0, self.max_buffer - self.current_data_size)
        return rwnd

# --- TEST ÇALIŞTIRMASI ---
receiver = TCPReceiverBuffer(max_buffer_size=4096)
SEGMENT_SIZE = 1024
APP_CONSUME_RATE = 256  # Yavaş okuyan uygulama

print("=" * 70)
print("TCP AKIŞ KONTROLÜ (FLOW CONTROL - RECEIVE WINDOW) SİMÜLASYONU")
print("Alıcı Tampon Kapasitesi: 4096 Bayt | Segment Boyutu: 1024 Bayt")
print("=" * 70)
print(f"{'Adım':<6} | {'Eylem':<22} | {'Tampon Yükü':<14} | {'Dönen rwnd':<12} | {'Durum'}")
print("-" * 70)

for adim in range(1, 7):
    # Gönderici veri basıyor
    rwnd = receiver.receive_segment(SEGMENT_SIZE)
    durum = "İletim Normal" if rwnd > 0 else "TAMPON DOLDU! STOP!"
    print(f"#{adim:<5} | Gönderici +{SEGMENT_SIZE}B Bastı | {receiver.current_data_size:<4} / 4096 B   | {rwnd:<6} Bayt | {durum}")
    
    if rwnd == 0:
        print("-" * 70)
        print("[!] KRİTİK: rwnd = 0! Gönderici veri basmayı kesti, Zero-Window Probe moduna geçti.")
        print("[*] Alıcı uygulama nihayet tamponu boşaltmaya başlıyor...")
        rwnd = receiver.application_read(2048)
        print(f"[*] Uygulama 2048B okudu -> Yeni rwnd Duyurusu: {rwnd} Bayt. Gönderici tekrar başlatıldı!")
        print("-" * 70)
        break
    
    # Uygulama yavaşça biraz veri okuyor
    receiver.application_read(APP_CONSUME_RATE)

---
### 6.3. Konsol Çıktısı ve Mühendislik Yorumu

```text
======================================================================
TCP AKIŞ KONTROLÜ (FLOW CONTROL - RECEIVE WINDOW) SİMÜLASYONU
Alıcı Tampon Kapasitesi: 4096 Bayt | Segment Boyutu: 1024 Bayt
======================================================================
Adım   | Eylem                  | Tampon Yükü    | Dönen rwnd   | Durum
----------------------------------------------------------------------
#1     | Gönderici +1024B Bastı | 1024 / 4096 B   | 3072   Bayt | İletim Normal
#2     | Gönderici +1024B Bastı | 1792 / 4096 B   | 2304   Bayt | İletim Normal
#3     | Gönderici +1024B Bastı | 2560 / 4096 B   | 1536   Bayt | İletim Normal
#4     | Gönderici +1024B Bastı | 3328 / 4096 B   | 768    Bayt | İletim Normal
#5     | Gönderici +1024B Bastı | 4096 / 4096 B   | 0      Bayt | TAMPON DOLDU! STOP!
----------------------------------------------------------------------
[!] KRİTİK: rwnd = 0! Gönderici veri basmayı kesti, Zero-Window Probe moduna geçti.
[*] Alıcı uygulama nihayet tamponu boşaltmaya başlıyor...
[*] Uygulama 2048B okudu -> Yeni rwnd Duyurusu: 2048 Bayt. Gönderici tekrar başlatıldı!
----------------------------------------------------------------------
```

#### Mühendislik Yorumu:
1. **Akış Kontrolü (Flow Control) vs Tıkanıklık Kontrolü (Congestion Control):** Bu simülasyonda görülen Akış Kontrolü; hızlı bir göndericinin yavaş bir **alıcıyı boğmasını (Receiver Buffer Overflow)** engeller. Tıkanıklık Kontrolü ise göndericinin **aradaki ağ yönlendiricilerini (Router Buffer Overflow)** tıkamasını engeller.
2. **Zero-Window Kilitlenmesi (Deadlock) ve Probe Çözümü:** Alıcı `rwnd = 0` duyurduğunda gönderici durur. Daha sonra alıcı tamponu boşaltıp `rwnd = 2048` paketini yolladığında bu bildirim yolda kaybolursa ne olur? İki taraf da birbirini sonsuza dek bekler (*Deadlock*). TCP bu sorunu çözmek için `rwnd = 0` iken göndericinin periyodik olarak 1 baytlık **Zero-Window Probe** paketleri atmasını zorunlu kılar; böylece güncel pencere boyutu er ya da geç öğrenilir.